In [0]:
# For this dataset, hashing is not required since there is no PII info, so we will just select the target columns accross the three datasets (base, variant 3 and variant 5), and merge them in a table
# I will start adding an identifier for each dataset.

from pyspark.sql.functions import monotonically_increasing_id, concat, lit

baf_base_silver = spark.table("fraud_bronze.baf_base")

baf_base_silver = baf_base_silver.withColumn(
    "account_id",
    concat(lit("baf_base_"), monotonically_increasing_id())
)

baf_variant_3_silver = spark.table('fraud_bronze.baf_variant_3')

baf_variant_3_silver = baf_variant_3_silver.withColumn(
    "account_id",
    concat(lit("baf_variant_3_"), monotonically_increasing_id())
)

baf_variant_5_silver = spark.table('fraud_bronze.baf_variant_5')

baf_variant_5_silver = baf_variant_5_silver.withColumn(
    "account_id",
    concat(lit("baf_variant_5_"), monotonically_increasing_id())
)

# Now, I will select the required columns for the models

baf_silver_col = [
    "account_id",
    "fraud_bool",
    "income",
    "name_email_similarity",
    "prev_address_months_count",
    "current_address_months_count",
    "customer_age",
    "days_since_request",
    "intended_balcon_amount",
    "payment_type",
    "zip_count_4w",
    "dataset_source"
]

# And now create the tables with the columns selected for each dataset

baf_silver_base_cols = baf_base_silver.select(baf_silver_col)
baf_silver_base_cols.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("fraud_silver.baf_base")

baf_silver_variant_3_cols = baf_variant_3_silver.select(baf_silver_col)
baf_silver_variant_3_cols.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("fraud_silver.baf_variant_3")

baf_silver_variant_5_cols = baf_variant_5_silver.select(baf_silver_col)
baf_silver_variant_5_cols.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("fraud_silver.baf_variant_5")


